In [16]:
import pandas as pd
import torch
import numpy as np


states = torch.load("states.pt")
target = pd.read_csv("target.csv")

In [63]:
DEVICE = torch.accelerator.current_accelerator() if torch.accelerator.is_available() else "cpu"

In [18]:
row = states.shape[0]

print(states.shape)
print(target)

torch.Size([707542, 30, 8, 8])
        value  policy
0          -1     307
1           1    4488
2          -1     657
3           1    4395
4          -1     195
...       ...     ...
707537      1    2538
707538     -1    4488
707539      1    2847
707540     -1    2604
707541      1    3480

[707542 rows x 2 columns]


In [49]:
import sys
sys.path.append('..')

In [66]:
policy = torch.tensor(target.policy.values).float()
value = torch.tensor(target.value.values).float()


In [ ]:
from core.network import PolicyValueNetwork
from torch import optim
import time
from core.network import PolicyValueNetwork
from torch import optim
import time
import torch
import numpy as np
from torch.optim import lr_scheduler


def train(network: PolicyValueNetwork,
          optimizer: optim.Optimizer,
          states: torch.Tensor,
          policy: torch.Tensor,
          value: torch.Tensor,
          policy_loss_fn,
          value_loss_fn,
          batch_size: int = 256,
          num_iter: int | None = None,
          duration_hour: float | None = None,
          seed: int = 42):

    if num_iter is None and duration_hour is None:
        raise ValueError("Must specify at least one of num_iter or duration_hour")

    start = time.time()
    rng = np.random.default_rng(seed=seed)
    step = 0

    # Make a LR Scheduler
    scheduler = lr_scheduler.CosineAnnealingLR(optimizer=optimizer, T_max=100)

    while True:
        if num_iter is not None and step >= num_iter:
            break
        if duration_hour is not None and time.time() - start >= duration_hour * 3600:
            break

        batch_idx = rng.choice(len(states), batch_size, replace=False)
        batch_states = states[batch_idx]
        batch_policy = policy[batch_idx]
        batch_value  = value[batch_idx]

        # Move policy to GPU
        batch_policy = batch_policy.to(device=DEVICE)
        batch_value = batch_value.to(device=DEVICE)

        optimizer.zero_grad()

        policy_head, value_head = network(batch_states)

        assert torch.allclose(policy.sum(dim=-1), torch.ones(len(policy)), atol=1e-5)

        policy_loss = policy_loss_fn(policy_head, batch_policy)
        value_loss  = value_loss_fn(value_head, batch_value)

        loss = policy_loss + value_loss
        loss.backward()

        optimizer.step()
        scheduler.step()

        if step % 10 == 0:
            elapsed = time.time() - start
            print(f"[{step}] loss={loss.item():.4f} | policy={policy_loss.item():.4f} | value={value_loss.item():.4f} | {elapsed:.0f}s")

        step += 1

In [73]:

from core import factory

network = factory.build_network("chess")

In [74]:
from torch.optim import Adam
from torch.nn import CrossEntropyLoss, MSELoss

optimizer = Adam(network.parameters(), lr=0.01, fused=True)
value_loss_fn = MSELoss()
policy_loss_fn = CrossEntropyLoss()

In [79]:
train(
    duration_hour=1,
    network=network,
    optimizer=optimizer,
    states=states, 
    policy=policy,
    value=value,
    policy_loss_fn=policy_loss_fn,
    value_loss_fn=value_loss_fn,
    batch_size=128,
    seed=42
)

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/torch/nn/modules/loss.py:610: UserWarning: Using a target size (torch.Size([128])) that is different to the input size (torch.Size([128, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


[0] loss=7.2104 | policy=6.6282 | value=0.5822 | 1s
[10] loss=6.7658 | policy=6.2409 | value=0.5249 | 16s
[20] loss=6.9699 | policy=6.4945 | value=0.4754 | 30s
[30] loss=6.7321 | policy=6.2939 | value=0.4382 | 45s
[40] loss=6.9629 | policy=6.4547 | value=0.5082 | 60s
[50] loss=6.7926 | policy=6.3417 | value=0.4509 | 82s
[60] loss=6.8910 | policy=6.3827 | value=0.5083 | 103s
[70] loss=6.7733 | policy=6.1252 | value=0.6482 | 117s
[80] loss=6.8359 | policy=6.2968 | value=0.5390 | 129s
[90] loss=6.7942 | policy=6.1922 | value=0.6020 | 141s
[100] loss=6.9421 | policy=6.3728 | value=0.5693 | 152s
[110] loss=6.7651 | policy=6.2484 | value=0.5167 | 167s
[120] loss=6.7164 | policy=6.2712 | value=0.4452 | 182s
[130] loss=6.9272 | policy=6.3491 | value=0.5781 | 197s
[140] loss=6.6891 | policy=6.1737 | value=0.5154 | 211s
[150] loss=7.0510 | policy=6.5269 | value=0.5240 | 226s
[160] loss=6.7636 | policy=6.2300 | value=0.5336 | 241s
[170] loss=6.8115 | policy=6.3136 | value=0.4979 | 255s
[180] loss